## Notebook for testing our pipeline for the Generative Recommender

In [1]:
# Imports here
from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
from rqvae_train import rqvae_loss, train_rqvae_sanity_check, train_rqvae_full, load_trained_rqvae
import torch
import numpy as np
import pickle
import os
import torch, numpy as np, random
import pandas as pd
from transformer import get_all_unique_sids, prepare_dataset, train_model, recommended_next_sid, is_model_trained
from transformers import BartTokenizer, BartForConditionalGeneration


# This is ONLY for our tests! Do not use for our full model
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

/Users/serkan/miniforge3/envs/generative_rec/lib/python3.10/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/serkan/miniforge3/envs/generative_rec/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <0B7EB158-53DC-3403-8A49-22178CAB4612> /Users/serkan/miniforge3/envs/generative_rec/lib/python3.10/site-packages/torchvision/image.so
  Reason: tried: '/Users/serkan/miniforge3/envs/generative_rec/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/serkan/miniforge3/envs/generative_rec/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/serkan/miniforge3/envs/generative_rec/lib/python3.10/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/serkan/miniforge3/envs/generative_rec/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvis

### First, get the feature strings from DataHandler

In [2]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [3]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

The embeddings have the shape: (105542, 384)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [5]:
input_dim = embeddings.shape[1]
latent_dim = 32
hidden_dim = 256
codebook_size = 512
num_quantizers = 4

TRAINING_MODE = 'None' # for loading

set_seed(42)
rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# trained_model = ...

print('Generating semantic IDs')

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()

if TRAINING_MODE == 'sanity_check':
    print("Running sanity check for RQ-VAE")
    rqvae = train_rqvae_sanity_check(rqvae, embeddings)

elif TRAINING_MODE == 'full_training':
    print("Training RQ-VAE")
    rqvae = train_rqvae_full(rqvae, embeddings, save_path='./models/rqvae')

else:
    rqvae = load_trained_rqvae(rqvae, 'quantizers/rqvae_training_results_20251101/models/rqvae_20251101_best.pth')

semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

print(f'semantic_ids has shape: {semantic_ids.shape}')
print('printing the first semantic ID (trained RQ-VAE)')
print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('quantizers/rqvae_training_results_20251101/item_2_semantic.pkl') and os.path.exists('quantizers/rqvae_training_results_20251101/semantic_2_item.pkl'):
    print('Loading existing hashmaps')

    with open('quantizers/rqvae_training_results_20251101/item_2_semantic.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('quantizers/rqvae_training_results_20251101/semantic_2_item.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:
    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')
    with open('item_2_semantic.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('semantic_2_item.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Generating semantic IDs
semantic_ids has shape: (105542, 4)
printing the first semantic ID (trained RQ-VAE)
[201  91 394  18]
Loading existing hashmaps
Loaded the hashmaps


### Converting transactions data: 
#### user_id: item_id1, ..., item_idn --> user_id: item_SID1, ..., item_SIDn

In [ ]:
transaction = pd.read_pickle('transactions_train.pkl')

customer_sid_sequences = {}
if os.path.exists('customer_sid_sequences.pkl'):
    print('Loading existing customer_sid_sequences')

    with open('customer_sid_sequences.pkl', 'rb') as f:
        customer_sid_sequences = pickle.load(f)
    print('Loaded existing customer_sid_sequences')
else:

    for customer_id, group in transaction.groupby("customer_id"):
        item_ids = group["article_id"].tolist()
        sid_list = [item_2_semantic[str(item_id)] for item_id in item_ids]
        customer_sid_sequences[customer_id] = sid_list
    
    print("customer_sid_sequences has been created, saving...")
    with open("customer_sid_sequences.pkl", "wb") as f:
        pickle.dump(customer_sid_sequences, f)
    print("Saving done!")

first_customer = list(customer_sid_sequences.keys())[0]
print(f"{first_customer}: {customer_sid_sequences[first_customer]}")

### Taking a subset of transaction data (customer_sid_sequences) for training of transfomer

In [ ]:
subset_keys = list(customer_sid_sequences.keys())[:100]
customer_transactions_subset = {k: customer_sid_sequences[k] for k in subset_keys}
print(customer_transactions_subset[list(customer_transactions_subset.keys())[0]])

### Training transformer with customer_transactions

In [ ]:
window_size=3

if is_model_trained(): 
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
    tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')
else: 
    print('There is no pretrained model, the model will be trained ...')

    unique_sids = get_all_unique_sids(customer_transactions_subset)
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
    tokenizer.add_tokens(unique_sids)

    model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
    # resize model embeddings after adding token to tokenizer
    model.resize_token_embeddings(len(tokenizer))

    dataset = prepare_dataset(customer_transactions_subset, window_size, tokenizer)
    train_model(dataset, model)

    model.save_pretrained('./bart-recommender/final_model')
    tokenizer.save_pretrained('./bart-recommender/final_model')

### Inference: recommend SIDs for given customer transaction sequence

In [ ]:
last_10_customers = list(customer_sid_sequences.keys())[-10:]
test_customers_sequences = {k: customer_sid_sequences[k] for k in last_10_customers}

for customer_id, seqs in test_customers_sequences.items():
    test_seq = test_customers_sequences[customer_id][-window_size:]
    test_seq = [' '.join(tuple(str(x) for x in sid)) for sid in test_seq]
    print(f"test SIDs sequence: {test_seq}")
    recommended_sids = recommended_next_sid(test_seq, model, tokenizer, window_size, top_k=12)
    filtered = [ # filter out empty sids
        (sid, semantic_2_item[tuple(int(x) for x in sid.split())])
        for sid in recommended_sids
        if sid.strip() != ''
    ]

    print("Recommended SIDs, generated by transformer: ")
    for sid, id in filtered:
        print(f'Rec. SID: {sid} --> article_id: {id}')